In [43]:
import pandas as pd

ts_df = pd.read_csv("../data/raw/traffic-signals.csv")

ts_df.head()

,id,mainStreet,side1Street,side2Street,controlMode,signalSystem,approachCount,audibleSignal,apsOperation,transitPreempt,...,bicycleSignal,ups,ledBlankoutSign,lpiNorthDate,lpiSouthDate,lpiEastDate,lpiWestDate,activationDate,latitude,longitude
0,2,JARVIS ST,FRONT ST E,NaN,PA - IPS,TransSuite,NaN,False,NaN,False,...,False,False,False,NaN,NaN,NaN,NaN,1948-11-15,43.647479,-79.371746
1,3,KING ST E,JARVIS ST,NaN,FT,TransSuite,4.0,True,Pushbutton Actuated,True,...,False,False,True,NaN,NaN,2021-06-23,2021-06-23,1950-08-23,43.650461,-79.371891
2,4,JARVIS ST,ADELAIDE ST E,NaN,SA1,TransSuite,3.0,False,NaN,False,...,True,False,False,NaN,NaN,NaN,NaN,1958-09-12,43.651527,-79.372365
3,5,JARVIS ST,RICHMOND ST E,NaN,SA1,TransSuite,3.0,False,NaN,False,...,False,True,False,2020-10-08,2020-10-08,NaN,NaN,1962-04-21,43.652704,-79.372831
4,6,JARVIS ST,QUEEN ST E,NaN,FT,TransSuite,4.0,False,NaN,False,...,False,False,False,2021-03-03,2021-03-03,2021-03-03,2021-03-03,1928-08-24,43.653729,-79.373254


In [44]:
ts_df.shape

(2562, 22)

In [45]:
ts_df.columns

Index(['id', 'mainStreet', 'side1Street', 'side2Street', 'controlMode',
       'signalSystem', 'approachCount', 'audibleSignal', 'apsOperation',
       'transitPreempt', 'firePreempt', 'railPreempt', 'bicycleSignal', 'ups',
       'ledBlankoutSign', 'lpiNorthDate', 'lpiSouthDate', 'lpiEastDate',
       'lpiWestDate', 'activationDate', 'latitude', 'longitude'],
      dtype='str')

In [46]:
ts_signals = ts_df[["id", "mainStreet", "side1Street", "side2Street", "controlMode", "activationDate"]]

ts_signals.head()

,id,mainStreet,side1Street,side2Street,controlMode,activationDate
0,2,JARVIS ST,FRONT ST E,NaN,PA - IPS,1948-11-15
1,3,KING ST E,JARVIS ST,NaN,FT,1950-08-23
2,4,JARVIS ST,ADELAIDE ST E,NaN,SA1,1958-09-12
3,5,JARVIS ST,RICHMOND ST E,NaN,SA1,1962-04-21
4,6,JARVIS ST,QUEEN ST E,NaN,FT,1928-08-24


In [47]:
ts_signals.controlMode.unique()

<StringArray>
['PA - IPS', 'FT', 'SA1', 'SA2', 'SAP', 'PA - MPS', 'SAV', 'FA', nan]
Length: 9, dtype: str

In [48]:
ts_signals.controlMode.value_counts(dropna=False)

controlMode
SA2         1305
SA1          471
SAP          363
FT           254
PA - MPS     101
SAV           38
NaN           16
PA - IPS      13
FA             1
Name: count, dtype: int64

In [49]:
signals_mask = ((ts_signals.controlMode != "PA - MPS") & (ts_signals.controlMode != "PA - IPS") & (ts_signals.controlMode.notna()))

ts_intersections = ts_signals[signals_mask].copy()

ts_intersections.controlMode.value_counts(dropna=False)

controlMode
SA2    1305
SA1     471
SAP     363
FT      254
SAV      38
FA        1
Name: count, dtype: int64

In [50]:
ts_intersections.shape

(2432, 6)

In [51]:
ts_intersections.activationDate = pd.to_datetime(ts_intersections.activationDate)

ts_intersections.activationDate.dtype

dtype('<M8[us]')

In [52]:
ts_intersections.activationDate.isna().sum()

np.int64(0)

In [53]:
mask_2018 = ts_intersections["activationDate"] <= pd.Timestamp("2018-09-01")

ts_intersections_2018 = ts_intersections[mask_2018].copy()

ts_intersections_2018.shape

(2258, 6)

In [54]:
ts_intersections_2018.id.is_unique

True

In [55]:
ksi_events = pd.read_csv("../data/interim/ksi_events.csv")

ksi_events.head()

,collision_id,accdate,stname1,stname2,stname3,acclass,accloc,traffictl,impactype,visible,light,rdsfcond,road_class,longitude,latitude,division,neighbourhood
0,2006:893184,2006-01-01 02:36:00,WOODBINE AVE,O CONNOR DR,NaN,Non-Fatal Injury,Intersection-Related,No Control,Approaching Head On,Clear,Dark,Wet,Major Arterial,-79.318798,43.699594,55 Division,Woodbine-Lumsden
1,2006:909646,2006-01-01 03:15:00,DANFORTH AVE,WEST LYNN AVE,NaN,Non-Fatal Injury,Intersection-Related,No Control,SMV Other,Snow,Dark with Artificial Lighting,Slush,Major Arterial,-79.316189,43.684873,55 Division,Woodbine Corridor
2,2026:6000734335,2026-04-10 06:10:00,ALBION RD,FINCH AV W,NaN,Non-Fatal Injury,Intersection-Related,Traffic Signal,SMV Other,Rain,Dark,Wet,NaN,-79.589200,43.741099,23 Division,Mount Olive-Silverstone-Jamestown
3,2026:6001470477,2026-07-15 18:33:00,28 CARNFORTH RD,LAWRENCE AV E,5 m East of,Non-Fatal Injury,Non-Intersection,Unknown,Approaching Head On,Clear,Dusk,Dry,NaN,-79.315404,43.739802,33 Division,Victoria Village
4,2026:6001482161,2026-07-17 09:39:00,HIGHWAY 2A E,MEADOWVALE RD,80 m East of,Non-Fatal Injury,Non-Intersection,No Control,Rear End,Clear,Daylight,Dry,NaN,-79.169880,43.780539,43 Division,Centennial Scarborough


In [56]:
strict_accloc = [
    "At Intersection",
    "Four-way",
    "T Intersection",
    "Other Intersection",
    "Roundabout",
]

strict_mask = (
    (ksi_events["traffictl"] == "Traffic Signal") &
    (ksi_events["accloc"].isin(strict_accloc))
)

strict_events = ksi_events[strict_mask].copy()

strict_events.shape

(2252, 17)

In [57]:
matchable_events = strict_events[
    strict_events["stname2"].notna()
].copy()

matchable_events.shape

(2233, 17)

In [58]:
matchable_events["stname1_normalized"] = (
    matchable_events["stname1"].str.upper().str.strip()
)

matchable_events[["stname1", "stname1_normalized"]].head()

,stname1,stname1_normalized
6,PARLIAMENT ST,PARLIAMENT ST
21,SPADINA AVE,SPADINA AVE
30,EGLINTON AVE W,EGLINTON AVE W
33,DUNDAS ST W,DUNDAS ST W
40,LAKE SHORE BV W,LAKE SHORE BV W


In [59]:
matchable_events["stname2_normalized"] = (
    matchable_events["stname2"].str.upper().str.strip()
)

matchable_events[["stname2", "stname2_normalized"]].head()

,stname2,stname2_normalized
6,FRONT ST E,FRONT ST E
21,NASSAU ST,NASSAU ST
30,BLACK CREEK DR,BLACK CREEK DR
33,427 C S DUNDAS RAMP S,427 C S DUNDAS RAMP S
40,JAMESON AV,JAMESON AV


In [60]:
ts_intersections_2018["mainStreet_normalized"] = (
    ts_intersections_2018["mainStreet"].str.upper().str.strip()
)

ts_intersections_2018[["mainStreet", "mainStreet_normalized"]].head()

,mainStreet,mainStreet_normalized
1,KING ST E,KING ST E
2,JARVIS ST,JARVIS ST
3,JARVIS ST,JARVIS ST
4,JARVIS ST,JARVIS ST
5,JARVIS ST,JARVIS ST


In [61]:
ts_intersections_2018["side1Street_normalized"] = (
    ts_intersections_2018["side1Street"].str.upper().str.strip()
)

ts_intersections_2018[["side1Street", "side1Street_normalized"]].head()

,side1Street,side1Street_normalized
1,JARVIS ST,JARVIS ST
2,ADELAIDE ST E,ADELAIDE ST E
3,RICHMOND ST E,RICHMOND ST E
4,QUEEN ST E,QUEEN ST E
5,SHUTER ST,SHUTER ST


In [62]:
ts_intersections_2018["side2Street_normalized"] = (
    ts_intersections_2018["side2Street"].str.upper().str.strip()
)

ts_intersections_2018.loc[
    ts_intersections_2018["side2Street"].notna(),
    ["side2Street", "side2Street_normalized"]
].head()

,side2Street,side2Street_normalized
11,JARVIS ST,JARVIS ST
13,WELLINGTON ST E,WELLINGTON ST E
36,COLLEGE ST,COLLEGE ST
39,DAVENPORT RD,DAVENPORT RD
40,BELMONT ST,BELMONT ST


In [63]:
matchable_events["intersection_key"] = matchable_events.apply(
    lambda row: " | ".join(sorted([
        row["stname1_normalized"],
        row["stname2_normalized"]
    ])),
    axis=1
)

matchable_events[[
    "stname1_normalized",
    "stname2_normalized",
    "intersection_key"
]].head()

,stname1_normalized,stname2_normalized,intersection_key
6,PARLIAMENT ST,FRONT ST E,FRONT ST E | PARLIAMENT ST
21,SPADINA AVE,NASSAU ST,NASSAU ST | SPADINA AVE
30,EGLINTON AVE W,BLACK CREEK DR,BLACK CREEK DR | EGLINTON AVE W
33,DUNDAS ST W,427 C S DUNDAS RAMP S,427 C S DUNDAS RAMP S | DUNDAS ST W
40,LAKE SHORE BV W,JAMESON AV,JAMESON AV | LAKE SHORE BV W


In [64]:
ts_intersections_2018["candidate_key_1"] = ts_intersections_2018.apply(
    lambda row: " | ".join(sorted([
        row["mainStreet_normalized"],
        row["side1Street_normalized"]
    ]))
    if pd.notna(row["mainStreet_normalized"]) and pd.notna(row["side1Street_normalized"])
    else pd.NA,
    axis=1
)

ts_intersections_2018[[
    "mainStreet_normalized",
    "side1Street_normalized",
    "candidate_key_1"
]].head()

,mainStreet_normalized,side1Street_normalized,candidate_key_1
1,KING ST E,JARVIS ST,JARVIS ST | KING ST E
2,JARVIS ST,ADELAIDE ST E,ADELAIDE ST E | JARVIS ST
3,JARVIS ST,RICHMOND ST E,JARVIS ST | RICHMOND ST E
4,JARVIS ST,QUEEN ST E,JARVIS ST | QUEEN ST E
5,JARVIS ST,SHUTER ST,JARVIS ST | SHUTER ST


In [65]:
ts_intersections_2018["candidate_key_2"] = ts_intersections_2018.apply(
    lambda row: " | ".join(sorted([
        row["mainStreet_normalized"],
        row["side2Street_normalized"]
    ]))
    if pd.notna(row["mainStreet_normalized"]) and pd.notna(row["side2Street_normalized"])
    else pd.NA,
    axis=1
)

ts_intersections_2018.loc[
    ts_intersections_2018["candidate_key_2"].notna(),
    ["mainStreet_normalized", "side2Street_normalized", "candidate_key_2"]
].head()

,mainStreet_normalized,side2Street_normalized,candidate_key_2
11,MOUNT PLEASANT RD,JARVIS ST,JARVIS ST | MOUNT PLEASANT RD
13,CHURCH ST,WELLINGTON ST E,CHURCH ST | WELLINGTON ST E
36,YONGE ST,COLLEGE ST,COLLEGE ST | YONGE ST
39,YONGE ST,DAVENPORT RD,DAVENPORT RD | YONGE ST
40,YONGE ST,BELMONT ST,BELMONT ST | YONGE ST


In [66]:

# ============================================================
# W04 — PANDAS DATASET PREPARATION
# Continue after candidate_key_2
# No scikit-learn code here
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. Prediction window
# ------------------------------------------------------------

start_date = pd.Timestamp("2018-01-01")
end_date = pd.Timestamp("2018-12-31")

signals = ts_intersections_2018.copy()
events = matchable_events.copy()


# ------------------------------------------------------------
# 2. Additional street-name normalization
# ------------------------------------------------------------

for df, columns in [
    (
        signals,
        [
            "mainStreet_normalized",
            "side1Street_normalized",
            "side2Street_normalized",
        ],
    ),
    (
        events,
        [
            "stname1_normalized",
            "stname2_normalized",
        ],
    ),
]:
    for col in columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.upper()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"\bAV\b", "AVE", regex=True)
            .str.replace(r"\bBV\b", "BLVD", regex=True)
        )


# ------------------------------------------------------------
# 3. Rebuild keys after normalization
# ------------------------------------------------------------

events["intersection_key"] = events[
    ["stname1_normalized", "stname2_normalized"]
].apply(
    lambda row: " | ".join(sorted(row.tolist()))
    if row.notna().all() else pd.NA,
    axis=1,
)

signals["candidate_key_1"] = signals[
    ["mainStreet_normalized", "side1Street_normalized"]
].apply(
    lambda row: " | ".join(sorted(row.tolist()))
    if row.notna().all() else pd.NA,
    axis=1,
)

signals["candidate_key_2"] = signals[
    ["mainStreet_normalized", "side2Street_normalized"]
].apply(
    lambda row: " | ".join(sorted(row.tolist()))
    if row.notna().all() else pd.NA,
    axis=1,
)


# ------------------------------------------------------------
# 4. Build a conservative signal-key reference
# ------------------------------------------------------------

primary_keys = (
    signals[
        ["id", "activationDate", "candidate_key_1"]
    ]
    .rename(columns={"candidate_key_1": "intersection_key"})
    .dropna(subset=["intersection_key"])
    .drop_duplicates(["id", "intersection_key"])
)

# Keep side2 candidates for diagnostics only.
secondary_keys = (
    signals[
        ["id", "candidate_key_2"]
    ]
    .rename(columns={"candidate_key_2": "intersection_key"})
    .dropna(subset=["intersection_key"])
    .drop_duplicates(["id", "intersection_key"])
)

key_id_counts = (
    primary_keys
    .groupby("intersection_key")["id"]
    .nunique()
)

unique_key_values = key_id_counts[key_id_counts == 1].index
ambiguous_key_values = key_id_counts[key_id_counts > 1].index

unique_signal_keys = (
    primary_keys[
        primary_keys["intersection_key"].isin(unique_key_values)
    ]
    .drop_duplicates("intersection_key")
)


# ------------------------------------------------------------
# 5. Match KSI events to signal IDs
# ------------------------------------------------------------

events["event_date"] = pd.to_datetime(
    events["accdate"],
    errors="coerce",
).dt.normalize()

events = events.merge(
    unique_signal_keys,
    on="intersection_key",
    how="left",
    validate="many_to_one",
)

events["match_status"] = "matched"

events.loc[
    events["id"].isna(),
    "match_status",
] = "no_key_match"

events.loc[
    events["id"].isna()
    & events["intersection_key"].isin(
        secondary_keys["intersection_key"]
    ),
    "match_status",
] = "side2_unverified"

events.loc[
    events["id"].isna()
    & events["intersection_key"].isin(
        ambiguous_key_values
    ),
    "match_status",
] = "ambiguous_key"

events.loc[
    events["intersection_key"].isna(),
    "match_status",
] = "missing_street"

events.loc[
    events["event_date"].isna(),
    "match_status",
] = "invalid_date"

events.loc[
    events["id"].notna()
    & events["event_date"].notna()
    & (events["event_date"] < events["activationDate"]),
    "match_status",
] = "inactive_at_event"


# ------------------------------------------------------------
# 6. Record unmatched events and reasons
# ------------------------------------------------------------

missing_street_events = strict_events.loc[
    strict_events["stname2"].isna(),
    ["collision_id", "accdate"],
].copy()

missing_street_events["intersection_key"] = pd.NA
missing_street_events["match_status"] = "missing_street"

unmatched_events = pd.concat(
    [
        events.loc[
            events["match_status"] != "matched",
            [
                "collision_id",
                "accdate",
                "intersection_key",
                "match_status",
            ],
        ],
        missing_street_events[
            [
                "collision_id",
                "accdate",
                "intersection_key",
                "match_status",
            ]
        ],
    ],
    ignore_index=True,
)

matching_report = (
    pd.concat(
        [
            events["match_status"],
            missing_street_events["match_status"],
        ],
        ignore_index=True,
    )
    .value_counts()
    .rename_axis("match_status")
    .reset_index(name="event_count")
)

matched_events = events.loc[
    events["match_status"] == "matched",
    ["id", "collision_id", "event_date"],
].copy()

matched_events = matched_events.rename(
    columns={"id": "intersection_id"}
)

matched_events["intersection_id"] = (
    matched_events["intersection_id"].astype("int64")
)

matched_events = matched_events.drop_duplicates(
    ["intersection_id", "collision_id", "event_date"]
)

print("KSI matching report:")
print(matching_report.to_string(index=False))


# ------------------------------------------------------------
# 7. Build independent intersection x day grid
# ------------------------------------------------------------

prediction_dates = pd.DataFrame(
    {
        "prediction_date": pd.date_range(
            start=start_date,
            end=end_date,
            freq="D",
        )
    }
)

observation_grid = (
    signals[["id", "activationDate"]]
    .rename(columns={"id": "intersection_id"})
    .merge(prediction_dates, how="cross")
)

observation_grid = observation_grid.loc[
    observation_grid["activationDate"]
    <= observation_grid["prediction_date"]
].copy()

observation_grid["target_date"] = (
    observation_grid["prediction_date"]
    + pd.Timedelta(days=1)
)


# ------------------------------------------------------------
# 8. Aggregate observed KSI by intersection and date
# ------------------------------------------------------------

daily_events = (
    matched_events
    .groupby(["intersection_id", "event_date"])["collision_id"]
    .nunique()
    .rename("daily_ksi")
    .reset_index()
)


# ------------------------------------------------------------
# 9. Build next-day binary target
# ------------------------------------------------------------

next_day_events = daily_events.rename(
    columns={
        "event_date": "target_date",
        "daily_ksi": "next_day_ksi",
    }
)

observation_grid = observation_grid.merge(
    next_day_events,
    on=["intersection_id", "target_date"],
    how="left",
    validate="one_to_one",
)

observation_grid["target"] = (
    observation_grid["next_day_ksi"]
    .fillna(0)
    .gt(0)
    .astype("int8")
)


# ------------------------------------------------------------
# 10. Historical KSI count before the prediction window
# ------------------------------------------------------------

history_before_window = (
    daily_events.loc[
        daily_events["event_date"] < start_date
    ]
    .groupby("intersection_id")["daily_ksi"]
    .sum()
    .rename("ksi_before_window")
)

observation_grid = observation_grid.merge(
    history_before_window,
    left_on="intersection_id",
    right_index=True,
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# 11. Daily history inside September 2018
# ------------------------------------------------------------

window_daily_events = (
    daily_events.loc[
        daily_events["event_date"].between(
            start_date,
            end_date,
        )
    ]
    .rename(columns={"event_date": "prediction_date"})
)

observation_grid = observation_grid.merge(
    window_daily_events,
    on=["intersection_id", "prediction_date"],
    how="left",
    validate="one_to_one",
)

observation_grid["ksi_before_window"] = (
    observation_grid["ksi_before_window"]
    .fillna(0)
    .astype("int64")
)

observation_grid["daily_ksi"] = (
    observation_grid["daily_ksi"]
    .fillna(0)
    .astype("int64")
)

observation_grid = observation_grid.sort_values(
    ["intersection_id", "prediction_date"]
).reset_index(drop=True)

observation_grid["historical_ksi_count"] = (
    observation_grid["ksi_before_window"]
    + observation_grid.groupby("intersection_id")[
        "daily_ksi"
    ].cumsum()
    - observation_grid["daily_ksi"]
)


# ------------------------------------------------------------
# 12. Calendar features
# ------------------------------------------------------------

# The weekday of the day being predicted is known in advance.
observation_grid["day_of_week"] = (
    observation_grid["target_date"].dt.dayofweek
)


# ------------------------------------------------------------
# 13. Build final pre-sklearn dataset
# ------------------------------------------------------------

final_dataset = observation_grid[
    [
        "intersection_id",
        "prediction_date",
        "target_date",
        "day_of_week",
        "historical_ksi_count",
        "target",
    ]
].copy()


# ------------------------------------------------------------
# 14. Essential checks
# ------------------------------------------------------------

duplicate_count = final_dataset.duplicated(
    ["intersection_id", "prediction_date"]
).sum()

missing_counts = final_dataset.isna().sum()

if duplicate_count != 0:
    raise ValueError(
        f"Duplicate intersection-day rows: {duplicate_count}"
    )

if missing_counts.sum() != 0:
    raise ValueError(
        f"Missing values found:\n{missing_counts}"
    )

positive_count = int(final_dataset["target"].sum())
positive_share = final_dataset["target"].mean()

print("\nFinal dataset:")
print("Shape:", final_dataset.shape)
print("Unique intersections:",
      final_dataset["intersection_id"].nunique())
print("Positive observations:", positive_count)
print("Positive share:", f"{positive_share:.4%}")

if positive_count == 0:
    print(
        "WARNING: no positive targets. "
        "Do not proceed to model training yet."
    )


# ------------------------------------------------------------
# 15. Save reproducible outputs
# ------------------------------------------------------------

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final_dataset.to_csv(
    output_dir / "intersection_day_2018_09.csv",
    index=False,
)

unmatched_events.to_csv(
    output_dir / "ksi_matching_exclusions.csv",
    index=False,
)

matching_report.to_csv(
    output_dir / "ksi_matching_report.csv",
    index=False,
)

print("\nPandas preparation finished.")
print(final_dataset.head(10))


KSI matching report:
     match_status  event_count
          matched         1706
     no_key_match          257
 side2_unverified          140
    ambiguous_key          122
   missing_street           19
inactive_at_event            8

Final dataset:
Shape: (823217, 6)
Unique intersections: 2258
Positive observations: 119
Positive share: 0.0145%

Pandas preparation finished.
   intersection_id prediction_date target_date  day_of_week  \
0                3      2018-01-01  2018-01-02            1   
1                3      2018-01-02  2018-01-03            2   
2                3      2018-01-03  2018-01-04            3   
3                3      2018-01-04  2018-01-05            4   
4                3      2018-01-05  2018-01-06            5   
5                3      2018-01-06  2018-01-07            6   
6                3      2018-01-07  2018-01-08            0   
7                3      2018-01-08  2018-01-09            1   
8                3      2018-01-09  2018-01-10      

In [67]:

# ============================================================
# W04 — FINAL DATASET DIAGNOSTICS
# Run after the Pandas preparation cell
# ============================================================

# 1. MATCHING QUALITY

print("=" * 60)
print("1. KSI MATCHING QUALITY")
print("=" * 60)

total_events = len(strict_events)
matched_count = matched_events["collision_id"].nunique()
matching_rate = matched_count / total_events * 100

print(f"Total strict KSI events: {total_events}")
print(f"Matched KSI events: {matched_count}")
print(f"Matching rate: {matching_rate:.2f}%")

print("\nMatching exclusions:")
print(matching_report.to_string(index=False))


# 2. MATCHED KSI EVENTS BY YEAR

print("\n" + "=" * 60)
print("2. MATCHED KSI EVENTS BY YEAR")
print("=" * 60)

yearly_ksi = (
    matched_events
    .assign(year=matched_events["event_date"].dt.year)
    .groupby("year")["collision_id"]
    .nunique()
    .rename("ksi_events")
)

print(yearly_ksi.to_string())


# 3. MATCHED KSI EVENTS BY MONTH IN 2018

print("\n" + "=" * 60)
print("3. MATCHED KSI EVENTS BY MONTH — 2018")
print("=" * 60)

events_2018 = matched_events.loc[
    matched_events["event_date"].dt.year == 2018
].copy()

monthly_ksi_2018 = (
    events_2018
    .groupby(events_2018["event_date"].dt.month)["collision_id"]
    .nunique()
    .reindex(range(1, 13), fill_value=0)
)

monthly_ksi_2018.index.name = "month"
monthly_ksi_2018.name = "ksi_events"

print(monthly_ksi_2018.to_string())


# 4. FINAL DATASET STRUCTURE

print("\n" + "=" * 60)
print("4. FINAL DATASET STRUCTURE")
print("=" * 60)

print("Shape:", final_dataset.shape)
print(
    "Unique intersections:",
    final_dataset["intersection_id"].nunique()
)
print(
    "Prediction date range:",
    final_dataset["prediction_date"].min(),
    "to",
    final_dataset["prediction_date"].max()
)
print(
    "Target date range:",
    final_dataset["target_date"].min(),
    "to",
    final_dataset["target_date"].max()
)

print("\nColumn types:")
print(final_dataset.dtypes.to_string())


# 5. TARGET DISTRIBUTION

print("\n" + "=" * 60)
print("5. TARGET DISTRIBUTION")
print("=" * 60)

target_counts = (
    final_dataset["target"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .sort_index()
)

positive_count = int(target_counts.loc[1])
positive_rate = final_dataset["target"].mean()

print(target_counts.to_string())
print(f"\nPositive rate: {positive_rate:.4%}")
print(f"Negative rate: {1 - positive_rate:.4%}")


# 6. POSITIVE OBSERVATIONS BY WEEK

print("\n" + "=" * 60)
print("6. POSITIVE OBSERVATIONS BY WEEK")
print("=" * 60)

weekly_positives = (
    final_dataset
    .groupby(
        final_dataset["target_date"].dt.to_period("W")
    )["target"]
    .sum()
)

print(weekly_positives.to_string())


# 7. ESSENTIAL DATA QUALITY CHECKS

print("\n" + "=" * 60)
print("7. DATA QUALITY CHECKS")
print("=" * 60)

required_columns = [
    "intersection_id",
    "prediction_date",
    "target_date",
    "day_of_week",
    "historical_ksi_count",
    "target",
]

duplicate_rows = final_dataset.duplicated(
    ["intersection_id", "prediction_date"]
).sum()

missing_values = (
    final_dataset[required_columns]
    .isna()
    .sum()
    .sum()
)

invalid_targets = (
    ~final_dataset["target"].isin([0, 1])
).sum()

negative_history = (
    final_dataset["historical_ksi_count"] < 0
).sum()

invalid_target_dates = (
    final_dataset["target_date"]
    != final_dataset["prediction_date"] + pd.Timedelta(days=1)
).sum()

collision_id_conflicts = (
    matched_events
    .groupby("collision_id")["intersection_id"]
    .nunique()
    .gt(1)
    .sum()
)

checks = {
    "No duplicate intersection-day rows": duplicate_rows == 0,
    "No missing required values": missing_values == 0,
    "Target contains only 0 and 1": invalid_targets == 0,
    "Historical KSI count is non-negative": negative_history == 0,
    "Target date is prediction date + 1 day": invalid_target_dates == 0,
    "Each matched collision has one intersection ID": collision_id_conflicts == 0,
}

for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check_name}")


# 8. FINAL DIAGNOSTIC SUMMARY

print("\n" + "=" * 60)
print("8. FINAL SUMMARY")
print("=" * 60)

print(f"Matching rate: {matching_rate:.2f}%")
print(f"Total observations: {len(final_dataset):,}")
print(f"Positive observations: {positive_count}")
print(f"Positive rate: {positive_rate:.4%}")
print(f"Basic data quality checks passed: {all(checks.values())}")

print("\nHistorical KSI date coverage:")
print(
    matched_events["event_date"].min(),
    "to",
    matched_events["event_date"].max()
)

if positive_count < 30:
    print(
        "\nWARNING: Very few positive observations. "
        "Consider extending the prediction period before CV."
    )

if not all(checks.values()):
    print(
        "\nWARNING: Data quality problems detected. "
        "Fix them before model training."
    )

print("\nDiagnostics completed.")


1. KSI MATCHING QUALITY
Total strict KSI events: 2252
Matched KSI events: 1706
Matching rate: 75.75%

Matching exclusions:
     match_status  event_count
          matched         1706
     no_key_match          257
 side2_unverified          140
    ambiguous_key          122
   missing_street           19
inactive_at_event            8

2. MATCHED KSI EVENTS BY YEAR
year
2006    113
2007     90
2008     96
2009     87
2010     95
2011     70
2012    137
2013     91
2014     83
2015     70
2016     88
2017    113
2018    119
2019    133
2020     82
2021     60
2022     72
2023     14
2024     26
2025     45
2026     22

3. MATCHED KSI EVENTS BY MONTH — 2018
month
1      4
2     10
3      4
4      8
5     10
6     10
7      9
8     12
9     14
10    16
11    18
12     4

4. FINAL DATASET STRUCTURE
Shape: (823217, 6)
Unique intersections: 2258
Prediction date range: 2018-01-01 00:00:00 to 2018-12-31 00:00:00
Target date range: 2018-01-02 00:00:00 to 2019-01-01 00:00:00

Column types:
in

In [68]:
print("KSI EVENTS BY YEAR (2016–2020)")
print(yearly_ksi.loc[2016:2020].to_string())

print("\nKSI EVENTS BY MONTH (2018)")
print(monthly_ksi_2018.to_string())

print("\nDATA QUALITY CHECKS")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

KSI EVENTS BY YEAR (2016–2020)
year
2016     88
2017    113
2018    119
2019    133
2020     82

KSI EVENTS BY MONTH (2018)
month
1      4
2     10
3      4
4      8
5     10
6     10
7      9
8     12
9     14
10    16
11    18
12     4

DATA QUALITY CHECKS
PASS — No duplicate intersection-day rows
PASS — No missing required values
PASS — Target contains only 0 and 1
PASS — Historical KSI count is non-negative
PASS — Target date is prediction date + 1 day
PASS — Each matched collision has one intersection ID


In [69]:
# ============================================================
# W04 — TEMPORAL TRAIN / VALIDATION SPLIT
# ============================================================

train_mask = (
    (final_dataset["prediction_date"] >= pd.Timestamp("2018-01-01")) &
    (final_dataset["prediction_date"] <= pd.Timestamp("2018-09-29"))
)

val_mask = (
    (final_dataset["prediction_date"] >= pd.Timestamp("2018-10-01")) &
    (final_dataset["prediction_date"] <= pd.Timestamp("2018-12-31"))
)

train_df = final_dataset.loc[train_mask].copy()
val_df = final_dataset.loc[val_mask].copy()

print("TRAIN")
print("Shape:", train_df.shape)
print("Positive targets:", train_df["target"].sum())
print("Positive share:", f"{train_df['target'].mean():.4%}")

print("\nVALIDATION")
print("Shape:", val_df.shape)
print("Positive targets:", val_df["target"].sum())
print("Positive share:", f"{val_df['target'].mean():.4%}")

TRAIN
Shape: (613223, 6)
Positive targets: 81
Positive share: 0.0132%

VALIDATION
Shape: (207736, 6)
Positive targets: 38
Positive share: 0.0183%


In [70]:
# ============================================================
# W04 — FEATURE / TARGET SEPARATION
# ============================================================

feature_columns = [
    "day_of_week",
    "historical_ksi_count"
]

# Training data
X_train = train_df[feature_columns].copy()
y_train = train_df["target"].copy()

# Validation data
X_val = val_df[feature_columns].copy()
y_val = val_df["target"].copy()

# Results
print("TRAIN")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nVALIDATION")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nFEATURES")
print(X_train.head())

TRAIN
X_train: (613223, 2)
y_train: (613223,)

VALIDATION
X_val: (207736, 2)
y_val: (207736,)

FEATURES
   day_of_week  historical_ksi_count
0            1                     0
1            2                     0
2            3                     0
3            4                     0
4            5                     0


In [73]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['day_of_week','historical_ksi_count']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [76]:
y_pred = model.predict(X_val)

pd.Series(y_pred).value_counts()

0    207736
Name: count, dtype: int64

In [77]:
y_proba = model.predict_proba(X_val)[:, 1]

pd.Series(y_proba).describe(
    percentiles=[0.5, 0.9, 0.99, 0.999]
)

count    207736.000000
mean          0.000192
std           0.000063
min           0.000158
50%           0.000164
90%           0.000265
99%           0.000444
99.9%         0.000947
max           0.000971
dtype: float64

In [78]:
ranking_df = pd.DataFrame({
    "target": y_val.to_numpy(),
    "risk_score": y_proba
})

top_n = int(len(ranking_df) * 0.01)

top_risk = ranking_df.nlargest(top_n, "risk_score")

print("Top 1% observations:", len(top_risk))
print("KSI in top 1%:", top_risk["target"].sum())
print("Total KSI:", ranking_df["target"].sum())

print(
    "Captured KSI:",
    f"{top_risk['target'].sum() / ranking_df['target'].sum():.2%}"
)

Top 1% observations: 2077
KSI in top 1%: 1
Total KSI: 38
Captured KSI: 2.63%


In [79]:
# Validation periods for temporal cross-validation

validation_periods = [
    ("Fold 1", "2018-05-01", "2018-06-30"),
    ("Fold 2", "2018-07-01", "2018-08-31"),
    ("Fold 3", "2018-09-01", "2018-09-29"),
]

for fold_name, start_date, end_date in validation_periods:

    mask = train_df["prediction_date"].between(
        pd.Timestamp(start_date),
        pd.Timestamp(end_date)
    )

    fold_df = train_df.loc[mask]

    positive_count = fold_df["target"].sum()

    print(f"{fold_name}:")
    print(f"  Observations: {len(fold_df)}")
    print(f"  Positive KSI: {positive_count}")
    print(f"  Positive share: {fold_df['target'].mean():.4%}")
    print()

Fold 1:
  Observations: 137464
  Positive KSI: 20
  Positive share: 0.0145%

Fold 2:
  Observations: 139939
  Positive KSI: 21
  Positive share: 0.0150%

Fold 3:
  Observations: 65482
  Positive KSI: 14
  Positive share: 0.0214%



In [80]:
# ============================================================
# W04 — TEMPORAL CROSS-VALIDATION: FOLD 1
# ============================================================

# Training: January 1 – April 29, 2018
fold1_train = train_df.loc[
    train_df["prediction_date"].between(
        pd.Timestamp("2018-01-01"),
        pd.Timestamp("2018-04-29")
    )
].copy()

# Validation: May 1 – June 30, 2018
fold1_val = train_df.loc[
    train_df["prediction_date"].between(
        pd.Timestamp("2018-05-01"),
        pd.Timestamp("2018-06-30")
    )
].copy()

# Results
print("FOLD 1 — TRAIN")
print("Shape:", fold1_train.shape)
print("Positive KSI:", fold1_train["target"].sum())

print("\nFOLD 1 — VALIDATION")
print("Shape:", fold1_val.shape)
print("Positive KSI:", fold1_val["target"].sum())

FOLD 1 — TRAIN
Shape: (268085, 6)
Positive KSI: 26

FOLD 1 — VALIDATION
Shape: (137464, 6)
Positive KSI: 20


In [81]:
# W04 — Fold 1: Feature / Target Separation

X_fold1_train = fold1_train[feature_columns].copy()
y_fold1_train = fold1_train["target"].copy()

print("X_fold1_train:", X_fold1_train.shape)
print("y_fold1_train:", y_fold1_train.shape)

X_fold1_train.head()

X_fold1_train: (268085, 2)
y_fold1_train: (268085,)


,day_of_week,historical_ksi_count
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [82]:
fold1_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

fold1_model.fit(X_fold1_train, y_fold1_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['day_of_week','historical_ksi_count']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [83]:
# W04 — Fold 1: Validation Features and Target

X_fold1_val = fold1_val[feature_columns].copy()
y_fold1_val = fold1_val["target"].copy()

print("X_fold1_val:", X_fold1_val.shape)
print("y_fold1_val:", y_fold1_val.shape)
print("Positive KSI:", y_fold1_val.sum())

X_fold1_val.head()

X_fold1_val: (137464, 2)
y_fold1_val: (137464,)
Positive KSI: 20


,day_of_week,historical_ksi_count
120,2,0
121,3,0
122,4,0
123,5,0
124,6,0


In [85]:
y_proba_fold1 = fold1_model.predict_proba(X_fold1_val)[:, 1]

pd.Series(y_proba_fold1).describe()

count    137464.000000
mean          0.000163
std           0.000030
min           0.000144
25%           0.000146
50%           0.000150
75%           0.000173
max           0.000491
dtype: float64

In [86]:
# W04 — Fold 1: Recall@1%

ranking_fold1 = pd.DataFrame({
    "target": y_fold1_val.to_numpy(),
    "risk_score": y_proba_fold1
})

top_n_fold1 = int(len(ranking_fold1) * 0.01)

top_risk_fold1 = ranking_fold1.nlargest(
    top_n_fold1,
    "risk_score"
)

total_ksi_fold1 = ranking_fold1["target"].sum()
captured_ksi_fold1 = top_risk_fold1["target"].sum()

recall_at_1_fold1 = captured_ksi_fold1 / total_ksi_fold1

print("FOLD 1 — RANKING EVALUATION")
print("Top 1% observations:", len(top_risk_fold1))
print("KSI in top 1%:", captured_ksi_fold1)
print("Total KSI:", total_ksi_fold1)
print("Recall@1%:", f"{recall_at_1_fold1:.2%}")

FOLD 1 — RANKING EVALUATION
Top 1% observations: 1374
KSI in top 1%: 0
Total KSI: 20
Recall@1%: 0.00%


In [87]:

# ============================================================
# W04 — TEMPORAL CROSS-VALIDATION: FOLD 2
# ============================================================

# 1. Temporal Split
# Train: January 1 — June 29
# Gap: June 30
# Validation: July 1 — August 31

fold2_train = train_df.loc[
    train_df["prediction_date"].between(
        pd.Timestamp("2018-01-01"),
        pd.Timestamp("2018-06-29")
    )
].copy()

fold2_val = train_df.loc[
    train_df["prediction_date"].between(
        pd.Timestamp("2018-07-01"),
        pd.Timestamp("2018-08-31")
    )
].copy()

print("FOLD 2 — DATA SPLIT")
print("Train:", fold2_train.shape)
print("Train positive KSI:", fold2_train["target"].sum())
print("Validation:", fold2_val.shape)
print("Validation positive KSI:", fold2_val["target"].sum())


# 2. Feature / Target Separation

X_fold2_train = fold2_train[feature_columns].copy()
y_fold2_train = fold2_train["target"].copy()

X_fold2_val = fold2_val[feature_columns].copy()
y_fold2_val = fold2_val["target"].copy()


# 3. Create a NEW Pipeline

fold2_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)


# 4. Train the model

fold2_model.fit(X_fold2_train, y_fold2_train)


# 5. Predict probabilities on validation

y_proba_fold2 = fold2_model.predict_proba(X_fold2_val)[:, 1]

print("\nFOLD 2 — PROBABILITY DISTRIBUTION")
print(pd.Series(y_proba_fold2).describe())


# 6. Ranking Evaluation — Recall@1%

ranking_fold2 = pd.DataFrame({
    "target": y_fold2_val.to_numpy(),
    "risk_score": y_proba_fold2
})

top_n_fold2 = int(len(ranking_fold2) * 0.01)

top_risk_fold2 = ranking_fold2.nlargest(
    top_n_fold2,
    "risk_score"
)

total_ksi_fold2 = ranking_fold2["target"].sum()
captured_ksi_fold2 = top_risk_fold2["target"].sum()

recall_at_1_fold2 = captured_ksi_fold2 / total_ksi_fold2

print("\nFOLD 2 — RANKING EVALUATION")
print("Top 1% observations:", len(top_risk_fold2))
print("KSI in top 1%:", captured_ksi_fold2)
print("Total KSI:", total_ksi_fold2)
print("Recall@1%:", f"{recall_at_1_fold2:.2%}")


# 7. Compare Fold 1 and Fold 2

print("\nCROSS-VALIDATION COMPARISON")
print(f"Fold 1 Recall@1%: {recall_at_1_fold1:.2%}")
print(f"Fold 2 Recall@1%: {recall_at_1_fold2:.2%}")


FOLD 2 — DATA SPLIT
Train: (405547, 6)
Train positive KSI: 46
Validation: (139939, 6)
Validation positive KSI: 21

FOLD 2 — PROBABILITY DISTRIBUTION
count    139939.000000
mean          0.000176
std           0.000046
min           0.000145
25%           0.000151
50%           0.000159
75%           0.000187
max           0.000732
dtype: float64

FOLD 2 — RANKING EVALUATION
Top 1% observations: 1399
KSI in top 1%: 1
Total KSI: 21
Recall@1%: 4.76%

CROSS-VALIDATION COMPARISON
Fold 1 Recall@1%: 0.00%
Fold 2 Recall@1%: 4.76%


In [88]:

# ============================================================
# W04 — TEMPORAL CROSS-VALIDATION: FOLD 3
# ============================================================

# 1. Temporal Split
# Train: January 1 — August 30
# Gap: August 31
# Validation: September 1 — September 29

fold3_train = train_df.loc[
    train_df["prediction_date"].between(
        pd.Timestamp("2018-01-01"),
        pd.Timestamp("2018-08-30")
    )
].copy()

fold3_val = train_df.loc[
    train_df["prediction_date"].between(
        pd.Timestamp("2018-09-01"),
        pd.Timestamp("2018-09-29")
    )
].copy()

print("FOLD 3 — DATA SPLIT")
print("Train:", fold3_train.shape)
print("Train positive KSI:", fold3_train["target"].sum())
print("Validation:", fold3_val.shape)
print("Validation positive KSI:", fold3_val["target"].sum())


# 2. Feature / Target Separation

X_fold3_train = fold3_train[feature_columns].copy()
y_fold3_train = fold3_train["target"].copy()

X_fold3_val = fold3_val[feature_columns].copy()
y_fold3_val = fold3_val["target"].copy()


# 3. Create a NEW Pipeline

fold3_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)


# 4. Train the model

fold3_model.fit(X_fold3_train, y_fold3_train)


# 5. Predict probabilities on validation

y_proba_fold3 = fold3_model.predict_proba(X_fold3_val)[:, 1]

print("\nFOLD 3 — PROBABILITY DISTRIBUTION")
print(pd.Series(y_proba_fold3).describe())


# 6. Ranking Evaluation — Recall@1%

ranking_fold3 = pd.DataFrame({
    "target": y_fold3_val.to_numpy(),
    "risk_score": y_proba_fold3
})

top_n_fold3 = int(len(ranking_fold3) * 0.01)

top_risk_fold3 = ranking_fold3.nlargest(
    top_n_fold3,
    "risk_score"
)

total_ksi_fold3 = ranking_fold3["target"].sum()
captured_ksi_fold3 = top_risk_fold3["target"].sum()

recall_at_1_fold3 = captured_ksi_fold3 / total_ksi_fold3

print("\nFOLD 3 — RANKING EVALUATION")
print("Top 1% observations:", len(top_risk_fold3))
print("KSI in top 1%:", captured_ksi_fold3)
print("Total KSI:", total_ksi_fold3)
print("Recall@1%:", f"{recall_at_1_fold3:.2%}")


# 7. Compare all three folds

print("\nCROSS-VALIDATION COMPARISON")
print(f"Fold 1 Recall@1%: {recall_at_1_fold1:.2%}")
print(f"Fold 2 Recall@1%: {recall_at_1_fold2:.2%}")
print(f"Fold 3 Recall@1%: {recall_at_1_fold3:.2%}")


FOLD 3 — DATA SPLIT
Train: (545483, 6)
Train positive KSI: 67
Validation: (65482, 6)
Validation positive KSI: 14

FOLD 3 — PROBABILITY DISTRIBUTION
count    65482.000000
mean         0.000184
std          0.000063
min          0.000153
25%          0.000154
50%          0.000155
75%          0.000201
max          0.000979
dtype: float64

FOLD 3 — RANKING EVALUATION
Top 1% observations: 654
KSI in top 1%: 1
Total KSI: 14
Recall@1%: 7.14%

CROSS-VALIDATION COMPARISON
Fold 1 Recall@1%: 0.00%
Fold 2 Recall@1%: 4.76%
Fold 3 Recall@1%: 7.14%


In [89]:
# ============================================================
# W04 — TEMPORAL CROSS-VALIDATION SUMMARY
# ============================================================

cv_results = pd.DataFrame({
    "fold": ["Fold 1", "Fold 2", "Fold 3"],

    "train_positives": [
        int(fold1_train["target"].sum()),
        int(fold2_train["target"].sum()),
        int(fold3_train["target"].sum())
    ],

    "val_positives": [
        int(fold1_val["target"].sum()),
        int(fold2_val["target"].sum()),
        int(fold3_val["target"].sum())
    ],

    "captured_ksi": [
        int(captured_ksi_fold1),
        int(captured_ksi_fold2),
        int(captured_ksi_fold3)
    ],

    "recall_at_1": [
        recall_at_1_fold1,
        recall_at_1_fold2,
        recall_at_1_fold3
    ]
})

# Aggregate results
mean_recall = cv_results["recall_at_1"].mean()

total_captured = cv_results["captured_ksi"].sum()
total_positives = cv_results["val_positives"].sum()

pooled_recall = total_captured / total_positives

# Output
print("W04 — TEMPORAL CROSS-VALIDATION SUMMARY")
print("=" * 60)

print(
    cv_results.to_string(
        index=False,
        formatters={
            "recall_at_1": "{:.2%}".format
        }
    )
)

print("\nAGGREGATED RESULTS")
print(f"Mean Recall@1%: {mean_recall:.2%}")
print(f"Captured KSI: {total_captured}/{total_positives}")
print(f"Pooled Recall@1%: {pooled_recall:.2%}")

W04 — TEMPORAL CROSS-VALIDATION SUMMARY
  fold  train_positives  val_positives  captured_ksi recall_at_1
Fold 1               26             20             0       0.00%
Fold 2               46             21             1       4.76%
Fold 3               67             14             1       7.14%

AGGREGATED RESULTS
Mean Recall@1%: 3.97%
Captured KSI: 2/55
Pooled Recall@1%: 3.64%
